# Figure 3: glyco-TWAS for AD

## 1. Setup

In [ ]:
library(data.table)
library(tidyverse)
library(ggrepel)
library(patchwork)
library(colocboost)

## 2. Load TWAS and MR results, exclude flagged glycopeptiforms

In [ ]:
twas_res_df <- fread('data_output/twas/glyco_v202601.twas_res.no_ashinf.csv.gz')

# exclude sole-modification Trp C-mannosylation glycopeptiforms mislabeled as glycation
qc_remove_gpids <- readLines('../../2_main_figures/Figure1/data/qc_remove_gpids.txt')
w_sole_exclude_gpids <- readLines('../../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt')
coding_drop_gpids <- readLines('../../2_main_figures/Figure1/data/coding_variant_drop_gpids.txt')
stopifnot(length(qc_remove_gpids) == 2, length(w_sole_exclude_gpids) == 54, length(coding_drop_gpids) == 21)
excluded_gpids <- c(qc_remove_gpids, w_sole_exclude_gpids, coding_drop_gpids)
w54_pattern <- paste0('_(', paste(excluded_gpids, collapse = '|'), ')$')
cat("twas_res_df rows matching excluded_gpids context suffix (to be dropped):",
    sum(str_detect(twas_res_df$context, w54_pattern)), "of", nrow(twas_res_df), "\n")
twas_res_df <- twas_res_df %>% filter(!str_detect(context, w54_pattern))

gene_ref <- fread('resource/references/Homo_sapiens.GRCh38.103.chr.reformatted.collapse_only.gene.region_list')

mr_res_df <- fread('data_output/mr/glyco_v202601.mr_res.csv.gz')

# exclude sole-modification Trp C-mannosylation glycopeptiforms mislabeled as glycation
cat("mr_res_df rows matching excluded_gpids context suffix (to be dropped):",
    sum(str_detect(mr_res_df$context, w54_pattern)), "of", nrow(mr_res_df), "\n")
mr_res_df <- mr_res_df %>% filter(!str_detect(context, w54_pattern))

mr_res_df_sig_unadj <- mr_res_df %>% filter(str_detect(context, 'DLPFC_Klein_gpQTL_unadjusted')) %>% .[
  meta_pval < 0.05 / .N &          # Bonferroni correction
  cpip >= 0.7 &                    # strong causal evidence
  num_CS >= 1 &                    # loose version (num_CS >= 1)
  Q_pval > 0.01 &                  # no heterogeneity
  I2 < 0.4                         # low heterogeneity
]

## 3. Significant glyco-TWAS and PWAS hits

### Unadjusted glyco-TWAS

In [ ]:
twas_unadj <- twas_res_df %>%filter(type == "unadjusted_gpQTL") 

n_imputable_genes <- twas_unadj %>%
  filter(is_imputable) %>%                  # keep only imputable rows
  distinct(molecular_id, context, gwas_study) %>%         # count unique gene-context pairs
  nrow()
n_imputable_genes

# Compute cutoff using Bonferroni-style correction
unadj_cutoff <- 0.05 / n_imputable_genes

twas_unadj_sig <- twas_unadj %>%
  mutate(sig = twas_pval < unadj_cutoff) %>%
  group_by(molecular_id, context, gwas_study) %>%
  summarise(
    n_methods = n(),
    n_sig = sum(sig, na.rm = T),
    best_sig = any(sig & is_selected_method, na.rm = T),
    .groups = "drop"
  ) %>%
  mutate(pass = (n_sig / n_methods >= 0.5) | best_sig) %>%
  mutate(pass_strict = (n_sig / n_methods >= 0.5)) %>%
  filter(pass) %>% left_join(twas_unadj) %>% arrange(twas_pval) %>% distinct(context, .keep_all = T)
twas_unadj_sig %>% dim

### Adjusted glyco-TWAS

In [ ]:
twas_adj <- twas_res_df %>% filter(type == "adjusted_gpQTL") 

n_imputable_genes <- twas_adj %>%
  filter(is_imputable) %>%                  # keep only imputable rows
  distinct(molecular_id, context, gwas_study) %>%         # count unique gene-context pairs
  nrow()
n_imputable_genes

### PWAS (pQTL)

In [ ]:
twas_pqtl <- twas_res_df %>% filter(type == "pQTL") 

n_imputable_genes <- twas_pqtl %>%
  filter(is_imputable) %>%                  # keep only imputable rows
  distinct(molecular_id, context, gwas_study) %>%         # count unique gene-context pairs
  nrow()
n_imputable_genes

# Compute cutoff using Bonferroni-style correction
pqtl_cutoff <- 0.05 / n_imputable_genes

twas_pqtl_sig <- twas_pqtl %>%
  mutate(sig = twas_pval < pqtl_cutoff) %>%
  group_by(molecular_id, context, gwas_study) %>%
  summarise(
    n_methods = n(),
    n_sig = sum(sig, na.rm = T),
    best_sig = any(sig & is_selected_method, na.rm = T),
    .groups = "drop"
  ) %>%
  mutate(pass = (n_sig / n_methods >= 0.5) | best_sig) %>%
  mutate(pass_strict = (n_sig / n_methods >= 0.5)) %>%
  filter(pass) %>% left_join(twas_pqtl) %>% arrange(twas_pval) %>% distinct(molecular_id, .keep_all = T)
twas_pqtl_sig %>% dim

## 4. ColocBoost AD colocalization

In [ ]:
gpQTL_colocalization <- data.table::fread("data_output/colocalization/maf05/glycoQTL_maf05/pQTL_glycoQTL_unadjusted_AD_summary_pv1e4.tsv")

# Trim excluded gpIDs (qc_remove_gpids + w_sole_exclude_gpids + coding_drop_gpids
strip_excl_tok <- function(s) {
  parts <- trimws(strsplit(s, ";")[[1]])
  gp_in_part <- str_extract(parts, "gp_[0-9]+")
  parts[is.na(gp_in_part) | !(gp_in_part %in% excluded_gpids)]
}
trimmed <- lapply(gpQTL_colocalization$colocalized_outcomes, strip_excl_tok)
keep_row <- lengths(trimmed) >= 2
cat(sprintf("gpQTL_colocalization: rows dropped entirely (<2 members after stripping excluded gpIDs): %d of %d\n", sum(!keep_row), nrow(gpQTL_colocalization)))
gpQTL_colocalization <- gpQTL_colocalization[keep_row]
gpQTL_colocalization$colocalized_outcomes <- vapply(trimmed[keep_row], paste, character(1), collapse = "; ")

gpQTL_colocalization <- gpQTL_colocalization %>% filter(str_detect(colocalized_outcomes,'AD_Bellenguez'))
colocboost_genes <- unique(gpQTL_colocalization$gene)

# glycopeptiform-level AD colocalization: gp contexts inside AD CoS (used to color Fig 3a)
colocboost_gp_contexts <- unique(unlist(strsplit(gpQTL_colocalization$colocalized_outcomes, ";\\s*")))
colocboost_gp_contexts <- colocboost_gp_contexts[str_detect(colocboost_gp_contexts, "gpQTL_unadjusted_gp_")]

## 5. Figure 3a: Manhattan plot

In [ ]:
df <- twas_unadj %>%
  mutate(sig = twas_pval < unadj_cutoff) %>%
  group_by(molecular_id, context, gwas_study) %>%
  summarise(
    n_methods = n(),
    n_sig = sum(sig, na.rm = TRUE),
    best_pval = min(twas_pval, na.rm = TRUE),
    best_method_pval = min(twas_pval[is_selected_method], na.rm = TRUE),
    best_method_selected = any(sig & is_selected_method, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    pass = (n_sig / n_methods >= 0.5) | best_method_selected, # pass defined by two threshold, best or half methods satisfied
    final_pval = ifelse(n_sig / n_methods >= 0.5, best_pval, best_method_pval),
    # # fallback: if no selected method existed
    final_pval = ifelse(is.infinite(final_pval), best_pval, final_pval)
  ) %>%
  left_join(twas_unadj, by = c("molecular_id", "context", "gwas_study")) %>%
  filter(twas_pval == final_pval) %>%
  distinct(context, .keep_all = TRUE) %>%
  mutate(
    logp = -log10(twas_pval) * sign(twas_z),
    pos = (start + end) / 2,
    is_coloc = context %in% colocboost_gp_contexts  # glycopeptiform level
  ) %>%
  merge(gene_ref, by.x = 'molecular_id', by.y = 'gene_id') %>%
  mutate(peptide = paste0(gene_name, ": ", str_extract(context, 'gp_.+$')))


df$chr <- factor(df$chr, levels = as.character(1:22))

chr_info <- df %>%
  group_by(chr) %>%
  summarise(chr_len = max(pos)) %>%
  mutate(chr_start = lag(cumsum(chr_len), default = 0))

df <- df %>%
  left_join(chr_info, by = "chr") %>%
  mutate(cum_pos = pos + chr_start) %>%
  mutate(
    TWAS_colocaliztion = ifelse(
      is_coloc & pass, 
      'TWAS & Colocalization',
      ifelse(!is_coloc & pass, 'TWAS only', 'Not Significant')
    )
  )
twas_unadj_sig_df = df
axis_df <- chr_info %>%
  mutate(center = chr_start + chr_len/2)

df <- mr_res_df_sig_unadj %>% rename('molecular_id' = 'gene_name') %>% merge(df, . , all.x = T)

In [ ]:
# Add a column for MR significance
df <- df %>%
  mutate(MR = ifelse(!is.na(Q_pval), "Significant", "Not Significant"))

dir.create('data', showWarnings = FALSE)
saveRDS(list(df = df, unadj_cutoff = unadj_cutoff, axis_df= axis_df), 'data/Figure_3a.rds')

## 6. Figure 3b: TWAS-confirmed AD-colocalized glycoproteins

In [ ]:
# Gene sets per analysis
unadj_genes <- unique(twas_unadj_sig$molecular_id)
pwas_genes <- unique(twas_pqtl_sig$molecular_id)

ad_glyco_genes   <- gpQTL_colocalization %>% filter(str_detect(colocalized_outcomes, 'gpQTL_unadjusted_gp_')) %>% pull(gene) %>% unique()
ad_protein_genes <- gpQTL_colocalization %>% filter(str_detect(colocalized_outcomes, 'pQTL_ENSG'))            %>% pull(gene) %>% unique()
ad_both_genes    <- intersect(ad_glyco_genes, ad_protein_genes)

all_genes <- list(
  `GlycoWAS`             = unadj_genes,
  `PWAS`                 = pwas_genes,
  `AD x Glyco`           = ad_glyco_genes,
  `AD x Protein`         = ad_protein_genes,
  `AD x Glyco x Protein` = ad_both_genes
)

# Step 1: get TWAS-hit genes
twas_genes <- twas_unadj_sig_df %>%
  filter(pass == TRUE) %>%
  distinct(molecular_id, gene_name)

coloc_filtered <- gpQTL_colocalization %>%
  filter(gene %in% twas_genes$molecular_id)

coloc_counts <- coloc_filtered %>%
  mutate(n_gp = str_count(colocalized_outcomes, "_gp_")) %>%
  group_by(gene) %>%
  summarise(n_coloc_gpforms = sum(n_gp), .groups = "drop")

coloc_counts_named <- coloc_counts %>%
  left_join(twas_genes, by = c("gene" = "molecular_id")) %>%
  arrange(desc(n_coloc_gpforms)) %>%
  mutate(gene_name = factor(gene_name, levels = unique(gene_name)))

In [ ]:
# --- Panel B right: two bars over AD-colocalized glycopeptiforms ---

ad_cos <- as.data.table(gpQTL_colocalization)[, cos_key := paste0(gene, ":", cos_id)]
bed <- fread(file.path("data_output/colocalization/maf05/glycoQTL_maf05", "pQTL_glycoQTL_unadjusted_AD_pv1e4.bed"))
setnames(bed, "cos_ID", "cos_key")
lead <- bed[bed[, .I[which.max(vcp)], by = cos_key]$V1]          # lead variant per CoS

dir_dt <- lead[cos_key %in% ad_cos$cos_key, {
  ev <- str_split(event_ID, ";\\s*")[[1]]
  cf <- as.numeric(str_split(coef, ";\\s*")[[1]])
  data.table(outcome = ev, coef = cf)
}, by = .(cos_key, gene = region_ID)]
dir_dt <- dir_dt[str_detect(outcome, "gpQTL_unadjusted_gp_")]
dir_dt[, gpID := str_extract(outcome, "gp_[0-9]+$")]

ad_dir <- lead[cos_key %in% ad_cos$cos_key, {
  ev <- str_split(event_ID, ";\\s*")[[1]]
  z  <- as.numeric(str_split(zscore, ";\\s*")[[1]])
  i  <- which(str_detect(ev, "AD_Bellenguez"))
  data.table(ad_z = if (length(i) == 1L) z[i] else NA_real_)
}, by = cos_key]
dir_dt <- merge(dir_dt, ad_dir, by = "cos_key", all.x = TRUE)

anno_ref <- fread("Annotation_with_gp_coordinates_clean.txt", sep = "\t")
dir_dt <- merge(dir_dt, unique(anno_ref[, .(gpID, GlycanClass, gene_name)]), by = "gpID", all.x = TRUE)
dir_dt[, GlycanClass := sub(";.*", "", GlycanClass)]
dir_dt[, direction := factor(ifelse(sign(coef) == sign(ad_z), "Positive", "Negative"), levels = c("Positive","Negative"))]

glycan_class_colors <- c("Paucimannose"="#A6CEE3","Hybrid/Complex"="#1F78B4",
                         "High-mannose"="#66C2A5","Glycation"="#C44E52","Other"="grey50")
glycan_class_labels <- c("Hybrid/Complex" = "Hybrid/complex")  # sentence case for legend display only
gene_order <- dir_dt[, .N, by = gene_name][order(-N)]$gene_name

int_breaks <- function(x) { b <- pretty(x); b[abs(b - round(b)) < 1e-9] }
bar_thm <- theme_minimal(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1), axis.text = element_text(size = 12),
        panel.grid = element_blank(), axis.line = element_line(linewidth = 0.3, color = "grey30"))

dir_twas <- dir_dt[gene %in% twas_genes$molecular_id]
gene_order_twas <- dir_twas[, .N, by = gene_name][order(-N)]$gene_name

dir_colors <- c(Positive = "#D55E00", Negative = "#0072B2")
div_dt <- dir_dt[, .(n = .N), by = .(gene_name, direction)]
div_dt[, y := ifelse(direction == "Negative", -n, n)]
f3b2_dir <- ggplot(div_dt, aes(x = factor(gene_name, levels = gene_order), y = y, fill = direction)) +
  geom_col(width = 0.7) +
  geom_hline(yintercept = 0, color = "grey40", linewidth = 0.4) +
  scale_fill_manual(values = dir_colors, name = "vs AD risk allele") +
  scale_y_continuous(breaks = int_breaks, labels = function(x) abs(x)) +
  labs(x = "Glycoprotein", y = "# glyco colocalized with AD\n(↑ up by risk allele  ·  ↓ down by risk allele)") + bar_thm

coloc_cnt <- dir_twas[, .N, by = .(gene_name, GlycanClass, direction)]
coloc_cnt[, y := ifelse(direction == "Negative", -N, N)]
f3b2_count <- ggplot(coloc_cnt,
                     aes(x = factor(gene_name, levels = gene_order_twas), y = y, fill = GlycanClass)) +
  geom_col(width = 0.7) +
  geom_hline(yintercept = 0, color = "grey40", linewidth = 0.4) +
  scale_fill_manual(values = glycan_class_colors, name = "Glycan class", labels = glycan_class_labels) +
  scale_y_continuous(breaks = int_breaks, labels = function(x) abs(x)) +
  labs(x = "Glycoprotein (TWAS-confirmed)",
       y = "# Colocalized glycopeptiforms") + bar_thm

In [ ]:
# Panel b: full glycopeptiform repertoire for the TWAS-confirmed AD-colocalized
pheno_v202609_ids <- fread(cmd = "zcat ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/phenotype_preprocessing/unadj_glyco.sorted.bed.gz | cut -f4")[[1]]
all_gp_dt <- unique(anno_ref[gene_name %in% gene_order_twas & gpID %in% pheno_v202609_ids & !gpID %in% excluded_gpids, .(gpID, gene_name, GlycanClass)])
all_gp_dt[, GlycanClass := sub(";.*", "", GlycanClass)]   # collapse compound classes to primary token
all_gp_dt <- all_gp_dt[, .N, by = .(gene_name, GlycanClass)]

f3b_all <- ggplot(all_gp_dt, aes(x = factor(gene_name, levels = gene_order_twas), y = N, fill = GlycanClass)) +
  geom_col(width = 0.7) +
  scale_fill_manual(values = glycan_class_colors, name = "Glycan class", labels = glycan_class_labels) +
  scale_y_continuous(breaks = int_breaks) +
  labs(x = "Glycoprotein (TWAS-confirmed)", y = "# Glycopeptiforms (all)") + bar_thm

saveRDS(list(all_genes = all_genes, coloc_counts_named = coloc_counts_named), 'data/Figure_3b.rds')
saveRDS(list(dir_dt = dir_dt, div_dt = div_dt, gene_order = gene_order, dir_twas = dir_twas, gene_order_twas = gene_order_twas,
             all_gp_dt = all_gp_dt,
             glycan_class_colors = glycan_class_colors,
             f3b2_count = f3b2_count, f3b2_dir = f3b2_dir, f3b_all = f3b_all),
        'data/Figure_3b_bars.rds')

library(patchwork)
options(repr.plot.width = 14, repr.plot.height = 6)
f3b2_dir | f3b2_count

## 7. Direction check: opposite-direction colocalized glycopeptiforms

In [ ]:
# Direction check: QTL effect direction taken directly from the colocalization bed

six_ids <- c(TMEM106B = "ENSG00000106460", ADAM10 = "ENSG00000137845",
             EGFR     = "ENSG00000146648", ACE    = "ENSG00000159640",
             SIRPA    = "ENSG00000198053", IGHG2  = "ENSG00000211893")

bed <- fread("data_output/colocalization/maf05/glycoQTL_maf05/pQTL_glycoQTL_unadjusted_AD_pv1e4.bed")
setnames(bed, "cos_ID", "cos_key")
lead <- bed[bed[, .I[which.max(vcp)], by = cos_key]$V1]            # lead variant per CoS

dir_res <- lead[region_ID %in% six_ids, {
  ev <- str_split(event_ID, ";\\s*")[[1]]
  data.table(outcome = ev,
             beta = as.numeric(str_split(coef,    ";\\s*")[[1]]),
             z    = as.numeric(str_split(zscore,  ";\\s*")[[1]]))
}, by = .(gene_id = region_ID, cos_id = cos_key, top_variant = variant_ID)]

dir_res[, gene_name := setNames(names(six_ids), six_ids)[gene_id]]
dir_res[, outcome := ifelse(str_detect(outcome, "gpQTL_unadjusted_gp_"),
                            str_extract(outcome, "gp_[0-9]+$"),
                            str_replace(outcome, "DLPFC_Bennett_pQTL_", "pQTL_"))]

gene_summary <- dir_res[str_detect(outcome, "^gp_[0-9]+$"), .(
  n_gp  = .N,
  n_pos = sum(z > 0, na.rm = TRUE),
  n_neg = sum(z < 0, na.rm = TRUE),
  opposite_direction = sum(z > 0, na.rm = TRUE) > 0 & sum(z < 0, na.rm = TRUE) > 0
), by = .(gene_name, gene_id, top_variant)]

print(gene_summary)
print(dir_res[order(gene_name), .(gene_name, outcome, beta = round(beta, 3), z = round(z, 2))])

dir.create('data', showWarnings = FALSE)
saveRDS(list(dir_res = dir_res, gene_summary = gene_summary),
        'data/Figure_3c_direction_check.rds')